# Textual surprise in FOMC communications: a tiny transformer study

**Author:** Francisco Salazar · **Course:** The AI Engineer, Week 3 capstone (v2) · **Design:** `DESIGN.md` (frozen; its version is `DESIGN_VERSION`, printed below)

**Abstract.** *(Phase 5 prose.)* Every number in this notebook is printed from variables.

In [ ]:
import time

T_START = time.perf_counter()
TIMES = {}  # stage -> seconds, reported in section 11

import dataclasses  # noqa: E402
import json  # noqa: E402
import math  # noqa: E402
import platform  # noqa: E402
import subprocess  # noqa: E402
import sys  # noqa: E402
from datetime import date  # noqa: E402
from pathlib import Path  # noqa: E402

import torch  # noqa: E402

MODE = "real"
REF = "week03-v2-run1"
DESIGN_VERSION = "v0.8"
# MODE: "real" is the only mode that scores real N/F. "rehearsal" (CUDA,
# full frozen settings) and "smoke" (CPU only, tiny settings) replace N and
# F by FAKE splits cut from T meetings; their numbers are not results.
MODES = ("real", "rehearsal", "smoke")
if MODE not in MODES:
    raise ValueError(f"MODE must be one of {MODES}")
FAKE_SPLIT = MODE != "real"
SMOKE = MODE == "smoke"
if torch.cuda.is_available():
    if SMOKE:
        raise RuntimeError("smoke mode is CPU-only; use MODE = 'rehearsal' "
                           "on a GPU")
    DEVICE = torch.device("cuda")
elif SMOKE:
    DEVICE = torch.device("cpu")
else:
    raise RuntimeError("CUDA GPU required: Runtime -> Change runtime type "
                       "-> T4 GPU, then Run all.")

REPO_URL = "https://github.com/FranQuant/the-ai-engineer.git"
SUBDIR = Path("capstones/week03_transformers")
MODULES = ("data.py", "bpe.py", "model.py", "evaluate.py", "ngram.py",
           "train.py", "analysis.py")


def has_modules(p: Path) -> bool:
    return all((p / m).is_file() for m in MODULES)


cwd = Path.cwd()
W3 = next((p.resolve() for p in (cwd, cwd.parent, cwd / SUBDIR,
                                 cwd / "the-ai-engineer" / SUBDIR)
           if has_modules(p)), None)
CODE_SOURCE = "local"  # "clone" when the modules come from the clone below
if W3 is None:
    dest = cwd / "the-ai-engineer"
    if dest.exists():
        raise RuntimeError(f"{dest} exists but lacks {MODULES}; remove it")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    REPO_URL, str(dest)], check=True)
    W3 = (dest / SUBDIR).resolve()
    if not has_modules(W3):
        raise RuntimeError(f"clone lacks {MODULES} under {SUBDIR}")
    CODE_SOURCE = "clone"


def uncommitted_changes(path: Path):
    """True if `git status --porcelain` lists changes under `path`, False if
    it lists none, None if git or the repository is unavailable."""
    try:
        out = subprocess.run(["git", "-C", str(path), "status",
                              "--porcelain", "--", "."],
                             capture_output=True, text=True)
    except FileNotFoundError:
        return None
    if out.returncode != 0:
        return None
    return bool(out.stdout.strip())


UNCOMMITTED_CHANGES = uncommitted_changes(W3)
if MODE == "real" and CODE_SOURCE == "local" and UNCOMMITTED_CHANGES:
    raise RuntimeError(f"real mode refuses local code with uncommitted "
                       f"changes in {W3}; commit them, or run from a "
                       f"clean clone of {REF}")
sys.path.insert(0, str(W3))

import analysis  # noqa: E402
import bpe  # noqa: E402
import data  # noqa: E402
import evaluate  # noqa: E402
import model  # noqa: E402
import ngram  # noqa: E402
import train  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

GIT_COMMIT = subprocess.run(["git", "-C", str(W3), "rev-parse", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
ENV = {"python": platform.python_version(), "torch": torch.__version__,
       "cuda": torch.version.cuda,
       "gpu": (torch.cuda.get_device_name(0) if DEVICE.type == "cuda"
               else f"none ({DEVICE.type})"),
       "ref": REF, "git_commit": GIT_COMMIT, "code_source": CODE_SOURCE,
       "uncommitted_changes": UNCOMMITTED_CHANGES, "mode": MODE,
       "design_version": DESIGN_VERSION}
for k, v in ENV.items():
    print(f"{k:>14}: {v}")
RUN_DIR = W3 / "runs" / MODE  # each mode has its own directory
FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
BANNER = ("#" * 72 + f"\n#  {MODE.upper()} RUN: fake N and F cut from T "
          "meetings. NOT A RESULT.\n" + "#" * 72)
if FAKE_SPLIT:
    print("\n" + BANNER)
TIMES["setup"] = time.perf_counter() - T_START

## 2. Introduction and hypotheses

*(Phase 5 prose.)* Confirmatory hypotheses (§6), char transformer unless stated:

- **H1 (drift):** Δ₁ = mean BPC(F) − mean BPC(N) > 0.
- **H2 (genre):** Δ₂ = mean over matched F meetings of [BPC(minutes) − BPC(statement)] > 0.
- **H3 (instrument robustness):** Spearman ρ(char, BPE) on F ≥ 0.7.

Exploratory (§7): E1, the five most surprising F documents; E2, ρ(char, n-gram) on F.

## 3. Data, normalization, split

The corpus, its manifest and the split manifest are checked against hard-coded SHA-256 values (§2, §9); a mismatch stops the notebook. Normalization is §3. The split is the committed meeting-level manifest; the §2 thresholds and the §3 character check are fail-closed (§10).

In [ ]:
t0 = time.perf_counter()
_docs = data.load_documents()  # SHA-256 + length checks; fails closed
split_manifest = data.load_split_manifest()
rules = split_manifest["rules"]
# Document IDs and real body sizes per split, from the manifest (no text).
MANIFEST_IDS = {sp: {e["document_id"] for m in split_manifest["meetings"]
                     if m["split"] == sp for e in m["documents"]}
                for sp in data.SPLITS}
REAL_CHARS = {k: sum(e["body_code_points"] for m in split_manifest["meetings"]
                     if m["split"] in sps for e in m["documents"])
              for k, sps in (("N+F", ("N", "F")), ("F", ("F",)))}

if FAKE_SPLIT:
    # N and F bodies are dropped right after the load and hash checks.
    t_only = [d for d in _docs if d.split == "T"]
    del _docs
    # FAKE split from T meetings only: the last 20% of T meetings as fake
    # F, the 10% before them as fake N, the rest as training T.
    t_meetings = sorted({d.meeting for d in t_only})
    n_fake_f = round(0.2 * len(t_meetings))
    n_fake_n = round(0.1 * len(t_meetings))
    fake_f = set(t_meetings[len(t_meetings) - n_fake_f:])
    fake_n = set(t_meetings[len(t_meetings) - n_fake_f - n_fake_n:
                            len(t_meetings) - n_fake_f])
    docs = [dataclasses.replace(
        d, split="F" if d.meeting in fake_f else
        "N" if d.meeting in fake_n else "T") for d in t_only]
    del t_only
    last_n = date.fromisoformat(max(fake_n))
    first_f = date.fromisoformat(min(fake_f))
    NF_BOUNDARY = last_n + (first_f - last_n) / 2
else:
    docs = [d for d in _docs if d.split in ("T", "N", "F")]
    del _docs
    NF_BOUNDARY = date.fromisoformat(rules["near_end"])
if data.check_characters(docs):  # §3: counts only, no text
    raise RuntimeError("INFEASIBLE (§10): N/F characters absent from T")

train_docs = [d for d in docs if d.split == "T"]
n_docs = [d for d in docs if d.split == "N"]
f_docs = [d for d in docs if d.split == "F"]


def split_counts(ds):
    by_meeting = {}
    for d in ds:
        by_meeting.setdefault(d.meeting, set()).add(d.genre)
    return {"documents": len(ds), "meetings": len(by_meeting),
            "statements": sum(d.genre == "statement" for d in ds),
            "minutes": sum(d.genre == "minutes" for d in ds),
            "matched_meetings": sum(g == {"statement", "minutes"}
                                    for g in by_meeting.values()),
            "body_chars": sum(len(d.body) for d in ds)}


COUNTS = {s: split_counts(ds) for s, ds in
          (("T", train_docs), ("N", n_docs), ("F", f_docs))}
print(f"{'split':>5} " + " ".join(f"{k:>16}" for k in COUNTS["T"]))
for s, c in COUNTS.items():
    print(f"{s:>5} " + " ".join(f"{v:>16,}" for v in c.values()))
print(f"N/F boundary for plots: {NF_BOUNDARY}")

# §2 feasibility thresholds (the real split only; the fake one is smaller)
FEASIBILITY = {"F documents >= 40": COUNTS["F"]["documents"] >= 40,
               "F matched pairs >= 15": COUNTS["F"]["matched_meetings"] >= 15,
               "N documents >= 16": COUNTS["N"]["documents"] >= 16}
if FAKE_SPLIT:
    print("§2 thresholds: not applied to the fake split")
else:
    for k, ok in FEASIBILITY.items():
        print(f"§2 {k}: {'pass' if ok else 'FAIL'}")
    if not all(FEASIBILITY.values()):
        raise RuntimeError("INFEASIBLE (§10): §2 thresholds fail")
print("§3 character check: pass")


def assert_scorable(ds, splits):
    """Documents (or their scores) are in `splits`. On a fake split every
    one is a real T document by manifest ID, never a real N or F one."""
    ds = list(ds)
    assert {d.split for d in ds} <= set(splits), "unexpected split"
    if FAKE_SPLIT:
        ids = {d.id for d in ds}
        assert not ids & (MANIFEST_IDS["N"] | MANIFEST_IDS["F"]), (
            "real N/F document on a fake split")
        assert ids <= MANIFEST_IDS["T"], "fake split holds a non-T document"


TIMES["data"] = time.perf_counter() - t0

## 4. Model and verification checks

The model code is imported from `model.py` (from scratch: scaled dot-product attention with boolean and additive masks, causal mask, self-attention, multi-head attention, FFN, Pre-LN block, sinusoidal positions, `TinyTransformerLM`). The §11 checks run here, on CPU, with visible output.

In [ ]:
t0 = time.perf_counter()
torch.set_printoptions(precision=4, sci_mode=False)
sdpa = model.scaled_dot_product_attention
CHECKS = {}  # every value printed below, saved in results.json

# §4.3 worked example
Q = torch.tensor([[1., 0], [0, 1], [1, 1]])
K = torch.tensor([[1., 0], [1, 1], [0, 1]])
V = torch.tensor([[1., 0], [0, 2], [3, 1]])
Y, S, A = sdpa(Q, K, V)
print("Q K^T =", Q @ K.T, "S = Q K^T / sqrt(2), shifted by the row max =",
      S, "A = softmax(S) =", A, "Y = A V =", Y, sep="\n")
r, e = 1 / math.sqrt(2), math.exp(-1 / math.sqrt(2))
assert torch.equal(Q @ K.T, torch.tensor([[1., 1, 0], [0, 1, 1], [1, 2, 1]]))
assert torch.allclose(S, torch.tensor([[0., 0, -r], [-r, 0, 0], [-r, 0, -r]]),
                      atol=1e-6)
A_ref = torch.tensor([[1, 1, e], [e, 1, 1], [e, 1, e]])
assert torch.allclose(A, A_ref / A_ref.sum(-1, keepdim=True), atol=1e-6)
assert torch.allclose(Y, torch.tensor([[0.994440, 1.0], [1.401112, 1.203336],
                                       [0.993020, 1.255235]]), atol=1e-5)
CHECKS["worked_example"] = {"QKT": (Q @ K.T).tolist(), "S": S.tolist(),
                            "A": A.tolist(), "Y": Y.tolist()}

# Causal rerun: token i attends to tokens <= i only.
Yc, _, Ac = sdpa(Q, K, V, mask=model.make_causal_mask(3))
print("causal A =", Ac, "causal Y =", Yc, sep="\n")
assert torch.equal(Ac[0], torch.tensor([1., 0, 0]))
assert torch.count_nonzero(torch.triu(Ac, diagonal=1)) == 0
Ya, _, _ = sdpa(Q, K, V, mask=model.make_causal_mask(3, additive=True))
assert torch.equal(Ya, Yc)  # boolean and additive masks agree
CHECKS["causal"] = {"A": Ac.tolist(), "Y": Yc.tolist()}

# Fused (PyTorch) vs manual attention
g = torch.Generator().manual_seed(0)
q, k, v = (torch.randn(2, 4, 16, 8, generator=g) for _ in range(3))
for causal in (False, True):
    manual, _, _ = sdpa(q, k, v, mask=model.make_causal_mask(16)
                        if causal else None)
    fused = torch.nn.functional.scaled_dot_product_attention(
        q, k, v, is_causal=causal)
    diff = (manual - fused).abs().max().item()
    print(f"fused vs manual (causal={causal}): max |diff| = {diff:.2e}")
    assert diff < 1e-5
    CHECKS[f"fused_vs_manual_max_diff_causal_{causal}"] = diff

# MHA with H = 1 equals single-head self-attention
torch.manual_seed(3)
sa = model.SelfAttention(d_model=4, causal=True)
mha = model.MultiHeadAttention(d_model=4, num_heads=1, causal=True)
with torch.no_grad():
    mha.proj_qkv.weight.copy_(torch.cat(
        [sa.W_Q.weight, sa.W_K.weight, sa.W_V.weight]))
    mha.proj_out.weight.copy_(torch.eye(4))
x = torch.randn(1, 5, 4)
diff = (mha(x) - sa(x)).abs().max().item()
print(f"MHA(H=1) vs self-attention: max |diff| = {diff:.2e}")
assert diff < 1e-6
CHECKS["mha_h1_vs_self_attention_max_diff"] = diff

# Uniform logits give loss log V
lm = model.TinyTransformerLM(model.ModelConfig(
    vocab_size=87, d_model=16, num_heads=2, num_layers=1, d_ff=32,
    block_size=8, dropout=0.0))
with torch.no_grad():
    lm.tok_emb.weight.zero_()  # tied head: every logit is 0
_, loss = lm(torch.randint(0, 87, (4, 8)), torch.randint(0, 87, (4, 8)))
print(f"uniform logits: loss {loss.item():.6f}, log V {math.log(87):.6f}")
assert abs(loss.item() - math.log(87)) < 1e-6
CHECKS["uniform_logits"] = {"loss": loss.item(), "log_V": math.log(87)}

# Trivial-pattern overfit: ABAB... is learned to near-zero loss
torch.manual_seed(1)
ab = torch.tensor([0, 1] * 200)
lm = model.TinyTransformerLM(model.ModelConfig(
    vocab_size=2, d_model=16, num_heads=2, num_layers=2, d_ff=32,
    block_size=8, dropout=0.0))
opt = torch.optim.Adam(lm.parameters(), lr=3e-3)
for _ in range(300):
    ix = torch.randint(0, len(ab) - 9, (16,))
    xb = torch.stack([ab[i:i + 8] for i in ix])
    yb = torch.stack([ab[i + 1:i + 9] for i in ix])
    _, loss = lm(xb, yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"AB overfit: loss after 300 steps {loss.item():.4f}")
assert loss.item() < 0.05
CHECKS["ab_overfit_loss_after_300_steps"] = loss.item()
del lm, mha, sa, opt
TIMES["checks"] = time.perf_counter() - t0
print("all §11 checks passed")

## 5. Training

Frozen §8 settings: C1 (d_model 256, 6 layers, 8 heads, d_ff 1024), block 256, batch 64, 3,696 steps per run; AdamW, lr 3e-4 with 200 warm-up steps then cosine to 3e-5; weight decay 0.1; dropout 0.1; clip 1.0; seed 1; fp16 autocast with GradScaler. Tokenizers and the transformers see T only. N is used for the training curves only (10 evaluations × 20 batches × 64 windows), which no decision reads.

In [ ]:
if SMOKE:
    ARCH = {"d_model": 64, "num_layers": 2, "num_heads": 4, "d_ff": 128}
    BLOCK_SIZE, BPE_VOCAB = 128, 500
    CFG = train.TrainConfig(steps=60, batch_size=8, warmup_steps=10,
                            monitor_batches=2, monitor_batch_size=8)
else:
    ARCH, BLOCK_SIZE, BPE_VOCAB = train.ARCH, train.BLOCK_SIZE, train.BPE_VOCAB
    CFG = train.TrainConfig()
_probe = train.make_model(10, BLOCK_SIZE, CFG.seed, **ARCH)
SETTINGS = {"arch": ARCH, "block_size": BLOCK_SIZE, "bpe_vocab": BPE_VOCAB,
            "train": dataclasses.asdict(CFG),
            "precision": "fp16" if DEVICE.type == "cuda" else "fp32",
            "dropout": _probe.blocks[0].ff.net[-1].p,
            "token_embedding_init_std": ARCH["d_model"] ** -0.5,
            "ngram_order": ngram.ORDER}
del _probe
print(json.dumps(SETTINGS, indent=1))

t0 = time.perf_counter()
char_vocab = data.CharVocab.from_documents(train_docs)
TIMES["char_vocab"] = time.perf_counter() - t0
t0 = time.perf_counter()
bpe_tok = bpe.SimpleBPE.from_documents(train_docs, BPE_VOCAB)
TIMES["bpe_fit"] = time.perf_counter() - t0
assert bpe_tok.vocab_size == BPE_VOCAB
TOKENIZERS = {"char": char_vocab, "bpe": bpe_tok}

t0 = time.perf_counter()
assert_scorable(n_docs, ("N",))
samplers, monitors, TOKEN_STATS = {}, {}, {}
for name, tok in TOKENIZERS.items():
    samplers[name] = data.WindowSampler.from_documents(train_docs, tok,
                                                       BLOCK_SIZE)
    monitors[name] = data.NMonitorSampler.from_documents(n_docs, tok,
                                                         BLOCK_SIZE)
    n_tokens = int(samplers[name].data.numel())
    TOKEN_STATS[name] = {
        "vocab_size": tok.vocab_size, "t_tokens_serialized": n_tokens,
        "tokens_per_char": (n_tokens - data.PREFIX_LEN * len(train_docs))
        / COUNTS["T"]["body_chars"],
        "passes_over_t": CFG.steps * CFG.batch_size * BLOCK_SIZE / n_tokens}
TIMES["tokenize"] = time.perf_counter() - t0
for name, s in TOKEN_STATS.items():
    print(f"{name:>4}: vocab {s['vocab_size']:,}, {s['t_tokens_serialized']:,}"
          f" T tokens ({s['tokens_per_char']:.4f}/char), "
          f"{s['passes_over_t']:.2f} passes over T")
print(f"BPE fit: {TIMES['bpe_fit']:.1f} s")

In [ ]:
MODELS, TRAINING = {}, {}


def run(name):
    tok = TOKENIZERS[name]
    net = train.make_model(tok.vocab_size, BLOCK_SIZE, CFG.seed, **ARCH)
    n_params = sum(p.numel() for p in net.parameters())
    init_std = net.tok_emb.weight.std().item()
    print(f"--- {name} transformer: {n_params:,} parameters, token "
          f"embedding init std {init_std:.4f}")
    res = train.train(net, samplers[name], monitors[name], CFG, DEVICE,
                      log_every=max(1, CFG.steps // 10))
    MODELS[name] = net.eval()
    TRAINING[name] = {"n_params": n_params,
                      "token_embedding_init_std_measured": init_std,
                      **dataclasses.asdict(res)}
    TIMES[f"train_{name}"] = res.seconds
    tail = res.train_loss[-50:]
    print(f"{name}: {res.steps:,} steps in {res.seconds:.1f} s "
          f"({res.precision}); mean train loss over the last {len(tail)} "
          f"steps {sum(tail) / len(tail):.4f} nats/token; "
          f"{res.skipped_updates} GradScaler-skipped updates (§8: counted, "
          f"not replaced)")


run("char")

In [ ]:
run("bpe")

In [ ]:
# One short sampling demo from the char model (§11): greedy, then T = 0.8.
t0 = time.perf_counter()
PROMPT = "The Committee"
prompt_ids = torch.tensor([data.serialize(char_vocab, "statement", PROMPT)],
                          device=DEVICE)
gen = torch.Generator(device=DEVICE).manual_seed(CFG.seed)
SAMPLES = {}
for label, kwargs in (("greedy", {"greedy": True}),
                      ("temperature 0.8", {"temperature": 0.8,
                                           "generator": gen})):
    out = MODELS["char"].generate(prompt_ids, 200, **kwargs)
    SAMPLES[label] = char_vocab.decode(out[0, data.PREFIX_LEN:].tolist())
    print(f"[{label}]\n{SAMPLES[label]}\n")
TIMES["sampling"] = time.perf_counter() - t0

## 6. Scoring protocol

§4: each document is scored on its own from `<BOS>` + genre, with sliding windows at stride block_size / 2; every body target is scored exactly once, in fp32, and the §4 assertions run per document. BPC = summed NLL / (body code points × ln 2). The char transformer scores N and F; the BPE transformer and the n-gram (Witten–Bell, order 5, fit on T) score F.

In [ ]:
assert_scorable(n_docs + f_docs, ("N", "F"))
t0 = time.perf_counter()
SCORES = {"char": evaluate.score_documents(MODELS["char"], char_vocab,
                                           n_docs + f_docs, BLOCK_SIZE)}
TIMES["score_char"] = time.perf_counter() - t0
t0 = time.perf_counter()
SCORES["bpe"] = evaluate.score_documents(MODELS["bpe"], bpe_tok, f_docs,
                                         BLOCK_SIZE)
TIMES["score_bpe"] = time.perf_counter() - t0
t0 = time.perf_counter()
ngram_lm = ngram.WittenBellNgram.from_documents(train_docs)
TIMES["ngram_fit"] = time.perf_counter() - t0
t0 = time.perf_counter()
SCORES["ngram"] = ngram_lm.score_documents(f_docs)
TIMES["score_ngram"] = time.perf_counter() - t0

for name, scores in SCORES.items():
    assert_scorable(scores, ("N", "F"))  # DocumentScore has id and split
    assert len({s.id for s in scores}) == len(scores)
if FAKE_SPLIT:
    print(f"{MODE} guard: every scored document is a real T document "
          "(manifest IDs); no real N/F document was scored")

MEAN_BPC = {name: {sp: float(np.mean([s.bpc for s in scores
                                      if s.split == sp]))
                   for sp in ("N", "F") if any(s.split == sp
                                               for s in scores)}
            for name, scores in SCORES.items()}
for name, means in MEAN_BPC.items():
    print(f"{name:>5}: " + ", ".join(
        f"{sp} {len([s for s in SCORES[name] if s.split == sp])} docs, "
        f"mean BPC {m:.4f}" for sp, m in means.items())
        + f"  ({TIMES['score_' + name]:.1f} s)")

# Rehearsal: scoring time projected to the real N+F (char) and F (BPE,
# n-gram) body sizes from the manifest, at the rate measured here.
SCORING_PROJECTION = None
if FAKE_SPLIT:
    fake_chars = {"N+F": COUNTS["N"]["body_chars"] + COUNTS["F"]["body_chars"],
                  "F": COUNTS["F"]["body_chars"]}
    SCORING_PROJECTION = {}
    for name, target in (("char", "N+F"), ("bpe", "F"), ("ngram", "F")):
        rate = TIMES[f"score_{name}"] / fake_chars[target]
        SCORING_PROJECTION[name] = {
            "target": target, "fake_chars": fake_chars[target],
            "real_chars": REAL_CHARS[target], "measured_s":
            TIMES[f"score_{name}"], "projected_s": rate * REAL_CHARS[target]}
        print(f"projected {name} scoring on real {target} "
              f"({REAL_CHARS[target]:,} chars): "
              f"{SCORING_PROJECTION[name]['projected_s']:.1f} s")

## 7. Results

Percentile bootstrap, 2,000 resamples of meetings, fixed seed (§6). Labels: **supported** if the 95% CI excludes 0 in the predicted direction; **contradicted** if the point estimate is ≤ 0; **inconclusive** otherwise. H3 is labelled by ρ ≥ 0.7 alone, CI reported.

In [ ]:
t0 = time.perf_counter()
H1 = analysis.h1_drift(SCORES["char"])
H2 = analysis.h2_genre(SCORES["char"])
H3 = analysis.h3_instruments(SCORES["char"], SCORES["bpe"])
E1 = analysis.e1_top(SCORES["char"])
E2 = analysis.e2_ngram(SCORES["char"], SCORES["ngram"])
TIMES["statistics"] = time.perf_counter() - t0


def show(name, est, what):
    print(f"{name}: {what} = {est.point:+.4f}, 95% CI "
          f"[{est.ci_low:+.4f}, {est.ci_high:+.4f}]"
          + (f" -> {est.label.upper()}" if est.label else ""))


show("H1", H1, "mean BPC(F) - mean BPC(N)")
print(f"    {H1.n_documents} documents in {H1.n_meetings} meetings")
show("H2", H2, "mean [BPC(minutes) - BPC(statement)]")
print(f"    {H2.n_meetings} matched F meetings; "
      f"{H2.excluded_meetings} unmatched F meetings excluded")
show("H3", H3, f"Spearman rho(char, BPE) on F (threshold "
     f"{analysis.H3_THRESHOLD})")
print(f"    {H3.n_documents} F documents; {H3.undefined_resamples} "
      f"undefined resamples")
print("E1: top five F documents by char BPC")
for i, d in enumerate(E1, 1):
    print(f"    {i}. {d['id']} ({d['genre']}, {d['meeting']}): "
          f"{d['bpc']:.4f}")
show("E2", E2, "Spearman rho(char, n-gram) on F")

In [ ]:
t0 = time.perf_counter()
# Reference palette, slots 1-2 (validated pair); text in ink tokens.
C1, C2 = "#2a78d6", "#eb6834"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "axes.edgecolor": INK2, "axes.labelcolor": INK,
    "axes.titlecolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "legend.frameon": False, "font.size": 10})
FIGURES = {}


def save(fig, name):
    if FAKE_SPLIT:
        fig.text(0.5, 0.5, f"{MODE.upper()}: NOT A RESULT", ha="center",
                 va="center", fontsize=28, color=INK2, alpha=0.25,
                 rotation=20)
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    FIGURES[name] = str(path)
    plt.show()


# Figure 1: training curves (T training loss, N monitoring) per instrument
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, name in zip(axes, ("char", "bpe")):
    tr = TRAINING[name]
    loss = np.array(tr["train_loss"])
    w = max(1, len(loss) // 50)
    smooth = np.convolve(loss, np.ones(w) / w, mode="valid")
    ax.plot(np.arange(w, len(loss) + 1), smooth, color=C1,
            label=f"T training loss ({w}-step mean)")
    ax.plot([m["step"] for m in tr["monitor"]],
            [m["loss"] for m in tr["monitor"]], color=C2, marker="o",
            markersize=6, label="N monitoring (plot only)")
    ax.set(title=f"{name} transformer", xlabel="step",
           ylabel=f"loss (nats per {name} token)")
axes[0].legend()
save(fig, "fig1_training_curves")

# Figure 2: per-document char BPC over time, N and F
fig, ax = plt.subplots(figsize=(10, 3.8))
for sp, color in (("N", C1), ("F", C2)):
    for genre, marker in (("statement", "o"), ("minutes", "s")):
        pts = [(date.fromisoformat(s.meeting), s.bpc) for s in SCORES["char"]
               if s.split == sp and s.genre == genre]
        if pts:
            ax.scatter(*zip(*pts), color=color, marker=marker, s=36,
                       edgecolors="white", linewidths=0.8,
                       label=f"{sp} {genre}")
ax.axvline(NF_BOUNDARY, color=INK2, linestyle="--", linewidth=1)
ax.annotate("N | F boundary", (NF_BOUNDARY, 1), xycoords=("data",
            "axes fraction"), xytext=(4, -12), textcoords="offset points",
            color=INK2, fontsize=9)
ax.set(title="Char transformer BPC per document", xlabel="meeting date",
       ylabel="bits per character")
ax.legend(ncol=4, loc="upper left", bbox_to_anchor=(0, -0.18))
save(fig, "fig2_bpc_over_time")

# Figure 3: paired genre plot over matched F meetings (H2)
pairs, _ = analysis.genre_pairs(SCORES["char"])
fig, ax = plt.subplots(figsize=(4.5, 4))
for _, m_bpc, s_bpc in pairs:
    ax.plot([0, 1], [s_bpc, m_bpc], color=GRID, linewidth=1, zorder=1)
ax.scatter([0] * len(pairs), [s for _, _, s in pairs], color=C1, s=36,
           zorder=2, label="statement")
ax.scatter([1] * len(pairs), [m for _, m, _ in pairs], color=C2, s=36,
           zorder=2, label="minutes")
ax.set(xticks=[0, 1], xticklabels=["statement", "minutes"], xlim=(-0.4, 1.4),
       title=f"Matched F meetings (n = {len(pairs)})",
       ylabel="char BPC")
ax.legend(loc="upper center")
save(fig, "fig3_genre_pairs")

# Figure 4: char vs BPE BPC on F (H3)
bpe_by_id = {s.id: s.bpc for s in SCORES["bpe"]}
fig, ax = plt.subplots(figsize=(4.8, 4.2))
for genre, color in (("statement", C1), ("minutes", C2)):
    pts = [(s.bpc, bpe_by_id[s.id]) for s in SCORES["char"]
           if s.split == "F" and s.genre == genre]
    ax.scatter(*zip(*pts), color=color, s=36, edgecolors="white",
               linewidths=0.8, label=genre)
ax.set(title=f"F documents: Spearman rho = {H3.point:.3f}",
       xlabel="char transformer BPC", ylabel="BPE transformer BPC")
ax.legend()
save(fig, "fig4_char_vs_bpe")
TIMES["figures"] = time.perf_counter() - t0

## 8. Limitations and future work

*(Phase 5 prose; see DESIGN.md §14.)*

## 9. References and reused code

*(Phase 5.)*

## 10. Use of AI tools

*(Phase 5.)*

## 11. Runtime

`runs/<MODE>/results.json` holds every number reported above. The total wall-clock time of this Run all is measured after that file is written, then added to it.

In [ ]:
RESULTS = {
    "design_version": DESIGN_VERSION, "mode": MODE, "environment": ENV,
    "settings": SETTINGS, "checks": CHECKS, "counts": COUNTS,
    "feasibility": None if FAKE_SPLIT else FEASIBILITY,
    "nf_boundary": str(NF_BOUNDARY), "tokenizers": TOKEN_STATS,
    "training": TRAINING,
    "skipped_updates": {k: v["skipped_updates"] for k, v in TRAINING.items()},
    "samples": {"prompt": PROMPT, **SAMPLES}, "mean_bpc": MEAN_BPC,
    "scores": {k: [dataclasses.asdict(s) for s in v]
               for k, v in SCORES.items()},
    "results": {"H1": H1.as_dict(), "H2": H2.as_dict(), "H3": H3.as_dict(),
                "E1": E1, "E2": E2.as_dict(),
                "bootstrap": {"resamples": analysis.RESAMPLES,
                              "seed": analysis.SEED,
                              "ci_level": analysis.CI_LEVEL}},
    "scoring_projection": SCORING_PROJECTION,
    "figures": FIGURES, "times_s": TIMES,
}
RESULTS_PATH = RUN_DIR / "results.json"
RESULTS_PATH.write_text(json.dumps(RESULTS, indent=1) + "\n")
TOTAL_S = time.perf_counter() - T_START  # after the full write
RESULTS["total_runtime_s"] = TOTAL_S
if SCORING_PROJECTION is not None:
    # Measured total with fake-split scoring swapped for the projection.
    # Fits on the smaller fake T (BPE, n-gram, tokenizing) are not rescaled.
    RESULTS["projected_real_total_s"] = TOTAL_S + sum(
        v["projected_s"] - v["measured_s"]
        for v in SCORING_PROJECTION.values())
RESULTS_PATH.write_text(json.dumps(RESULTS, indent=1) + "\n")
for k, v in TIMES.items():
    print(f"{k:>12}: {v:7.1f} s")
print(f"saved {RESULTS_PATH}")
print(f"TOTAL RUNTIME: {TOTAL_S:.1f} s ({TOTAL_S / 60:.2f} min)")
if "projected_real_total_s" in RESULTS:
    print(f"projected real-run total (scoring rescaled to real N/F sizes): "
          f"{RESULTS['projected_real_total_s']:.1f} s "
          f"({RESULTS['projected_real_total_s'] / 60:.2f} min)")
if FAKE_SPLIT:
    print(BANNER)